# Public release note

This is a sanitized public copy of `omniASR_20_K6_KenLM_V6_final_inference(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# 20.K6 — Inférence test finale KenLM V6

Ce notebook applique **exactement** le package ayant obtenu le PASS de `20.K5` aux 94 shards / 41 733 clips du test Kaggle. Il utilise la base BF16, le LoRA Maasai uniquement sur `mas|unscripted`, le batch acoustique 1 et un seul décodeur KenLM résident.

Il crée un CSV uniquement après une QA bloquante. **Il ne soumet rien automatiquement à Kaggle et ne remplace pas la configuration active.**

## Ordre d'exécution

1. Choisir un GPU **A100** (recommandé) ou **L4** (suffisant).
2. Exécuter **20.K6.A**. Si Colab demande un redémarrage, redémarrer puis rejouer A.
3. Exécuter **20.K6.B** et le laisser terminer. En cas de coupure, relancer la même cellule : les shards complets et les points partiels de 50 lignes sont repris.
4. Exécuter **20.K6.C** pour vérifier le PASS et récupérer le chemin du CSV.

Estimation prudente : **8–20 h sur A100**, **12–30 h sur L4**. Une ETA réelle est affichée toutes les 100 nouvelles lignes.

In [ ]:
# 20.K6.A — Setup Colab GPU idempotent
# Choisir d'abord Runtime > Modifier le type d'exécution > L4 ou A100.
from google.colab import drive
drive.mount('/content/drive')

import importlib.metadata as md
import json, os, shutil, subprocess, sys
from pathlib import Path

assert shutil.which("nvidia-smi"), "Choisir un runtime GPU L4 ou A100 avant cette cellule."

def version_or_none(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

loaded_before = {name for name in ("torch", "numpy") if name in sys.modules}
before = {name: version_or_none(name) for name in ("torch", "numpy")}

# Reprendre le commit et les versions critiques figés par le PASS 20.K5.
k5_root = Path("/content/afrivoices_project/finetune_runs/experiment_run/reports/kenlm_v6_20_K5")
k5_latest = json.load(open(k5_root / "LATEST_PASS.json", encoding="utf-8"))
assert k5_latest["run_id"] == "REPLACE_WITH_LOCAL_RUN_ID"
assert k5_latest["status"] == "PASS_READY_FOR_20_K6_TEST_INFERENCE"
k5_contract_doc = json.load(
    open(Path(k5_latest["report"]).parent / "contract.json", encoding="utf-8")
)
k5_software = k5_contract_doc["software_fingerprint"]
commit = k5_software["omnilingual_asr_git_commit"]
assert len(commit) == 40

repo = "/content/omnilingual-asr-k6"
if not os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run([
        "git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/facebookresearch/omnilingual-asr.git", repo,
    ], check=True)
subprocess.run(["git", "-C", repo, "fetch", "--depth", "1", "origin", commit], check=True)
subprocess.run(["git", "-C", repo, "checkout", "--detach", commit], check=True)

for executable in ("ffmpeg", "rsync"):
    if not shutil.which(executable):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", executable], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    repo,
    "pyarrow>=20.0.0", "pandas>=2.2.0", "numpy<3",
    "soundfile>=0.12.1", "pyctcdecode>=0.5.0", "kenlm>=0.3.0",
    "psutil>=5.9.0",
], check=True)

# Rejouer les versions observées par K5. Le commit git fixe omnilingual-asr ;
# torch est aligné sur sa version publique sans dépendre du suffixe CUDA local.
expected_torch = k5_software["packages"].get("torch")
if expected_torch and expected_torch != "UNKNOWN":
    expected_torch_base = expected_torch.split("+")[0]
    current_torch = version_or_none("torch")
    if not current_torch or current_torch.split("+")[0] != expected_torch_base:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"torch=={expected_torch_base}"],
            check=True,
        )

critical = (
    "fairseq2", "numpy", "pandas", "pyarrow", "soundfile",
    "pyctcdecode", "kenlm", "psutil",
)
pins = []
for package in critical:
    expected = k5_software["packages"].get(package)
    if expected and expected != "UNKNOWN" and version_or_none(package) != expected:
        pins.append(f"{package}=={expected}")
if pins:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins], check=True)

# Aligner les extensions binaires Colab avec torch.
torch_base = md.version("torch").split("+")[0]
torch_mm = ".".join(torch_base.split(".")[:2])
tv_for_torch = {
    "2.6": "0.21.0", "2.7": "0.22.0", "2.8": "0.23.0",
    "2.9": "0.24.0", "2.10": "0.25.0", "2.11": "0.26.0"
}
targets = {"torchaudio": torch_base, "torchvision": tv_for_torch.get(torch_mm)}
realigned = False
for package, target in targets.items():
    if not target:
        continue
    current = version_or_none(package)
    if not current or current.split("+")[0] != target:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"{package}=={target}"],
            check=True,
        )
        realigned = True

after = {name: version_or_none(name) for name in ("torch", "numpy")}
changed_loaded = [
    name for name in loaded_before
    if before[name] and after[name]
    and before[name].split("+")[0] != after[name].split("+")[0]
]
if changed_loaded or realigned:
    print("⚠️ Des bibliothèques binaires ont été réalignées.")
    print("Runtime > Redémarrer la session, puis réexécuter 20.K6.A et 20.K6.B.")
    raise SystemExit("REDÉMARRAGE COLAB REQUIS")

actual_commit = subprocess.run(
    ["git", "-C", repo, "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert actual_commit == commit

for package in critical:
    expected = k5_software["packages"].get(package)
    if expected and expected != "UNKNOWN":
        assert version_or_none(package) == expected, (
            package, version_or_none(package), expected,
            "Redémarrer la session puis rejouer 20.K6.A."
        )
expected_omni = k5_software["packages"].get("omnilingual-asr")
if expected_omni and expected_omni != "UNKNOWN":
    assert version_or_none("omnilingual-asr") == expected_omni
if expected_torch and expected_torch != "UNKNOWN":
    assert version_or_none("torch").split("+")[0] == expected_torch.split("+")[0]

import torch
assert torch.cuda.is_available(), "Le runtime n'a pas de GPU CUDA."
assert torch.cuda.is_bf16_supported(), "Le GPU doit prendre BF16 en charge."
print("✅ Setup 20.K6 prêt")
print("GPU :", torch.cuda.get_device_name(0), "| BF16 :", torch.cuda.is_bf16_supported())
print("torch :", md.version("torch"), "| fairseq2 :", md.version("fairseq2"))
print("omnilingual-asr commit :", actual_commit)


## 20.K6.B — Inventaire, inférence reprenable, assemblage et QA

La cellule commence par vérifier cryptographiquement `20.K5`, ses poids, ses 12 routes et les 94 shards. Elle ne garde qu'un shard test sur le SSD local à la fois.

In [ ]:
K6_WORKER_SOURCE = '"""Worker GPU reprenable pour 20.K6.\n\nIl ne retune rien. Il applique exactement le runtime K5, traite un shard test a\nla fois, et publie uniquement des caches de predictions signes sur Drive.\n"""\n\nfrom __future__ import annotations\n\nimport base64\nimport ctypes\nimport gc\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport unicodedata\nfrom collections import OrderedDict\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport psutil\nimport pyarrow.parquet as pq\nimport soundfile as sf\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nassert len(sys.argv) == 3, "usage: worker.py INPUT_JSON RESULT_JSON"\nINPUT_PATH = Path(sys.argv[1]).resolve()\nRESULT_PATH = Path(sys.argv[2]).resolve()\nSPEC = json.loads(INPUT_PATH.read_text(encoding="utf-8"))\n\n\ndef sha256_file(path: Path, block_size: int = 16 << 20) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(block_size), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef canonical_json(value: Any) -> str:\n    return json.dumps(\n        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str\n    )\n\n\ndef canonical_sha256(value: Any) -> str:\n    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()\n\n\ndef sequence_digest(values: list[str]) -> str:\n    digest = hashlib.sha256()\n    for value in values:\n        encoded = str(value).encode("utf-8")\n        digest.update(len(encoded).to_bytes(8, "big"))\n        digest.update(encoded)\n    return digest.hexdigest()\n\n\ndef atomic_json(value: Any, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")\n    with temporary.open("w", encoding="utf-8") as handle:\n        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)\n    os.replace(temporary, path)\n\n\ndef atomic_parquet(frame: pd.DataFrame, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")\n    frame.to_parquet(temporary, index=False)\n    os.replace(temporary, path)\n\n\ndef emit(event: str, **values: Any) -> None:\n    print(\n        "K6_EVENT " + json.dumps(\n            {"event": event, "elapsed_seconds": time.monotonic() - STARTED, **values},\n            ensure_ascii=False,\n        ),\n        flush=True,\n    )\n\n\ndef malloc_trim() -> None:\n    gc.collect()\n    try:\n        ctypes.CDLL("libc.so.6").malloc_trim(0)\n    except Exception:\n        pass\n\n\nSTARTED = time.monotonic()\nRUN_ID = str(SPEC["run_id"])\nCONTRACT_SHA256 = str(SPEC["contract_sha256"])\nCACHE_ROOT = Path(SPEC["cache_root"]).resolve()\nLOCAL_STAGE_ROOT = Path(SPEC["local_stage_root"]).resolve()\nBASE_WEIGHT = Path(SPEC["base_weight"]).resolve()\nADAPTER_WEIGHT = Path(SPEC["adapter_weight"]).resolve()\nLORA_PLAN_PATH = Path(SPEC["lora_plan"]).resolve()\nRUNTIME_CONFIG_PATH = Path(SPEC["runtime_config"]).resolve()\nTEST_MANIFEST_PATH = Path(SPEC["test_manifest"]).resolve()\n\nfor path, expected in (\n    (BASE_WEIGHT, SPEC["base_weight_sha256"]),\n    (ADAPTER_WEIGHT, SPEC["adapter_weight_sha256"]),\n    (LORA_PLAN_PATH, SPEC["lora_plan_sha256"]),\n    (RUNTIME_CONFIG_PATH, SPEC["runtime_config_sha256"]),\n    (TEST_MANIFEST_PATH, SPEC["test_manifest_sha256"]),\n):\n    assert path.is_file(), path\n    assert sha256_file(path) == expected, (path.name, "sha256")\n\nBASE_PARAMETERS = int(SPEC["base_parameters"])\nADAPTER_PARAMETERS = int(SPEC["adapter_parameters"])\nTOTAL_PARAMETERS = int(SPEC["total_parameters"])\nassert BASE_PARAMETERS + ADAPTER_PARAMETERS == TOTAL_PARAMETERS\nassert TOTAL_PARAMETERS < 1_000_000_000\nassert int(SPEC["batch_size"]) == 1\n\nassert torch.cuda.is_available(), "20.K6 requiert un GPU L4/A100."\nassert torch.cuda.is_bf16_supported(), "Le GPU doit prendre BF16 en charge."\ntorch.manual_seed(int(SPEC["seed"]))\ntorch.cuda.manual_seed_all(int(SPEC["seed"]))\ntorch.backends.cuda.matmul.allow_tf32 = False\ntorch.backends.cudnn.allow_tf32 = False\ntorch.backends.cudnn.benchmark = False\ntorch.cuda.reset_peak_memory_stats()\n\nRUNTIME_CONFIG = json.loads(RUNTIME_CONFIG_PATH.read_text(encoding="utf-8"))\nassert set(RUNTIME_CONFIG) == set(SPEC["runtime_groups"])\nTEST_RECORDS = json.loads(TEST_MANIFEST_PATH.read_text(encoding="utf-8"))\nassert isinstance(TEST_RECORDS, list) and len(TEST_RECORDS) == 94\nassert sum(int(row["rows"]) for row in TEST_RECORDS) == 41_733\n\nCACHE_ROOT.mkdir(parents=True, exist_ok=True)\nLOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)\n\n\ndef cache_paths(record: dict[str, Any]) -> tuple[Path, Path]:\n    token = hashlib.sha256(record["relative_path"].encode("utf-8")).hexdigest()[:12]\n    stem = Path(record["relative_path"]).stem\n    base = CACHE_ROOT / record["submission_language"] / record["domain"]\n    output = base / f"{stem}_{token}.parquet"\n    return output, output.with_suffix(output.suffix + ".meta.json")\n\n\ndef partial_cache_paths(record: dict[str, Any]) -> tuple[Path, Path]:\n    output, _ = cache_paths(record)\n    partial = output.with_suffix(output.suffix + ".partial.parquet")\n    return partial, partial.with_suffix(partial.suffix + ".meta.json")\n\n\ndef base_shard_spec(record: dict[str, Any]) -> dict[str, Any]:\n    return {\n        "run_id": RUN_ID,\n        "contract_sha256": CONTRACT_SHA256,\n        "relative_path": record["relative_path"],\n        "submission_language": record["submission_language"],\n        "manifest_language": record["manifest_language"],\n        "domain": record["domain"],\n        "source_bytes": int(record["bytes"]),\n        "source_sha256": record["source_sha256"],\n        "rows": int(record["rows"]),\n        "ids_sha256": record["ids_sha256"],\n    }\n\n\ndef valid_cache(record: dict[str, Any]) -> tuple[bool, dict[str, Any] | None]:\n    output, meta_path = cache_paths(record)\n    if not output.is_file() or not meta_path.is_file():\n        return False, None\n    try:\n        meta = json.loads(meta_path.read_text(encoding="utf-8"))\n        assert meta["status"] == "PASS_SHARD_CACHE"\n        assert meta["run_id"] == RUN_ID\n        assert meta["contract_sha256"] == CONTRACT_SHA256\n        assert meta["base_spec_sha256"] == canonical_sha256(base_shard_spec(record))\n        assert meta["source_sha256"] == record["source_sha256"]\n        assert int(meta["batch_size"]) == 1\n        assert bool(meta["adapter_enabled"]) == (\n            record["group"] == "mas|unscripted"\n        )\n        assert meta["cache_sha256"] == sha256_file(output)\n        frame = pd.read_parquet(output)\n        assert list(frame.columns) == ["id", "language", "transcription"]\n        ids = frame["id"].astype(str).tolist()\n        assert len(frame) == int(record["rows"])\n        assert len(ids) == len(set(ids))\n        assert sequence_digest(ids) == record["ids_sha256"]\n        assert frame["language"].astype(str).eq(record["submission_language"]).all()\n        assert meta["rows"] == len(frame)\n        return True, meta\n    except Exception:\n        return False, None\n\n\ncache_records: list[dict[str, Any]] = []\nmissing: list[dict[str, Any]] = []\ncached_clips = 0\ncached_decode_errors = 0\nfor record in TEST_RECORDS:\n    valid, meta = valid_cache(record)\n    if valid:\n        output, meta_path = cache_paths(record)\n        cache_records.append(\n            {\n                "relative_path": record["relative_path"],\n                "output": str(output),\n                "meta": str(meta_path),\n                "cache_sha256": meta["cache_sha256"],\n                "source_sha256": meta["source_sha256"],\n                "rows": int(record["rows"]),\n                "reused": True,\n            }\n        )\n        cached_clips += int(record["rows"])\n        cached_decode_errors += int(meta.get("decode_errors", 0))\n    else:\n        missing.append(record)\n\nemit(\n    "cache_preflight",\n    shards_total=len(TEST_RECORDS),\n    shards_cached=len(TEST_RECORDS) - len(missing),\n    shards_missing=len(missing),\n    clips_cached=cached_clips,\n    audio_decode_errors_cached=cached_decode_errors,\n)\nassert cached_decode_errors <= int(SPEC["max_decode_errors"]), (\n    cached_decode_errors,\n    SPEC["max_decode_errors"],\n)\n\n\ndef projection_dimensions(module: nn.Module) -> tuple[int, int] | None:\n    weight = getattr(module, "weight", None)\n    if not isinstance(weight, torch.Tensor) or weight.ndim != 2:\n        return None\n    name = type(module).__name__.lower()\n    namespace = type(module).__module__.lower()\n    if any(token in name for token in ("embedding", "embed", "conv", "norm")):\n        return None\n    if any(True for _ in module.children()):\n        return None\n    input_dim = getattr(module, "in_features", getattr(module, "input_dim", None))\n    output_dim = getattr(module, "out_features", getattr(module, "output_dim", None))\n    input_dim = int(input_dim) if input_dim is not None else int(weight.shape[1])\n    output_dim = int(output_dim) if output_dim is not None else int(weight.shape[0])\n    if tuple(weight.shape) != (output_dim, input_dim):\n        return None\n    if not (\n        isinstance(module, nn.Linear)\n        or "linear" in name\n        or "projection" in name\n        or "projection" in namespace\n    ):\n        return None\n    return input_dim, output_dim\n\n\nclass EvalLoRAProjection(nn.Module):\n    def __init__(self, base: nn.Module, rank: int):\n        super().__init__()\n        dimensions = projection_dimensions(base)\n        assert dimensions is not None\n        input_dim, output_dim = dimensions\n        self.base = base\n        self.rank = int(rank)\n        self.scaling = 1.0\n        self.enabled = False\n        self.lora_A = nn.Parameter(\n            torch.zeros(\n                self.rank,\n                input_dim,\n                device=base.weight.device,\n                dtype=base.weight.dtype,\n            ),\n            requires_grad=False,\n        )\n        self.lora_B = nn.Parameter(\n            torch.zeros(\n                output_dim,\n                self.rank,\n                device=base.weight.device,\n                dtype=base.weight.dtype,\n            ),\n            requires_grad=False,\n        )\n\n    def forward(self, inputs: torch.Tensor, *args: Any, **kwargs: Any) -> torch.Tensor:\n        output = self.base(inputs, *args, **kwargs)\n        if not self.enabled:\n            return output\n        return output + F.linear(F.linear(inputs, self.lora_A), self.lora_B) * self.scaling\n\n\ndef parent_and_child(root: nn.Module, dotted_name: str) -> tuple[nn.Module, str]:\n    parts = dotted_name.split(".")\n    parent = root\n    for part in parts[:-1]:\n        parent = getattr(parent, part)\n    return parent, parts[-1]\n\n\ndef labels_from_tokenizer(tokenizer: Any) -> list[str]:\n    tokenizer_model = getattr(tokenizer, "_model", None)\n    for method in ("index_to_token", "id_to_piece", "IdToPiece"):\n        if tokenizer_model is not None and hasattr(tokenizer_model, method):\n            labels = [\n                str(getattr(tokenizer_model, method)(index))\n                for index in range(tokenizer.vocab_info.size)\n            ]\n            labels[0] = ""\n            assert len(labels) == tokenizer.vocab_info.size\n            assert len(set(labels)) == len(labels)\n            return labels\n    raise RuntimeError("Vocabulaire tokenizer inaccessible")\n\n\nmodel = None\npipe = None\nlabels: list[str] | None = None\nlabels_sha256: str | None = None\nwrappers: OrderedDict[str, EvalLoRAProjection] = OrderedDict()\ndecoder = None\ndecoder_group = None\nmodel_loaded = False\nmax_resident_decoders = 0\nresident_decoders = 0\n\n\ndef load_runtime() -> None:\n    global model, pipe, labels, labels_sha256, wrappers, model_loaded\n    if model_loaded:\n        return\n    emit("model_load_start", gpu=torch.cuda.get_device_name(0))\n    card_name = str(SPEC["model_card_name"])\n    package_spec = importlib.util.find_spec("omnilingual_asr")\n    assert package_spec is not None and package_spec.submodule_search_locations\n    package_root = Path(next(iter(package_spec.submodule_search_locations))).resolve()\n    card_dir = package_root / "cards/models"\n    card_dir.mkdir(parents=True, exist_ok=True)\n    card_path = card_dir / f"{card_name}.yaml"\n    card_path.write_text(\n        f"name: {card_name}\\nbase: omniASR_CTC_1B_v2\\n"\n        f"checkpoint: {json.dumps(\'file://\' + str(BASE_WEIGHT))}\\n",\n        encoding="utf-8",\n    )\n\n    from fairseq2.data.tokenizers.hub import load_tokenizer\n    from fairseq2.models.hub import load_model\n    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline\n\n    model = load_model(card_name, device=torch.device("cuda"), dtype=torch.bfloat16)\n    model.eval()\n    assert sum(parameter.numel() for parameter in model.parameters()) == BASE_PARAMETERS\n\n    plan = pd.read_parquet(LORA_PLAN_PATH)\n    assert len(plan) == int(SPEC["lora_plan_rows"])\n    assert int(plan["parameters_per_adapter"].sum()) == ADAPTER_PARAMETERS\n    for row in plan.sort_values("module").to_dict("records"):\n        parent, child = parent_and_child(model, str(row["module"]))\n        base = getattr(parent, child)\n        assert projection_dimensions(base) == (\n            int(row["in_features"]), int(row["out_features"])\n        )\n        wrapper = EvalLoRAProjection(base, int(row["rank"]))\n        setattr(parent, child, wrapper)\n        wrappers[str(row["module"])] = wrapper\n    assert len(wrappers) == int(SPEC["lora_plan_rows"])\n    assert sum(parameter.numel() for parameter in model.parameters()) == TOTAL_PARAMETERS\n\n    raw_adapter = torch.load(ADAPTER_WEIGHT, map_location="cpu", weights_only=False)\n    assert raw_adapter.get("language") == "mas"\n    assert int(raw_adapter.get("step", -1)) == 1250\n    assert raw_adapter.get("plan_sha256") == SPEC["lora_plan_sha256"]\n    state = raw_adapter.get("state_dict", raw_adapter)\n    parameters = dict(model.named_parameters())\n    expected_keys = {\n        f"{module}.lora_A" for module in wrappers\n    } | {\n        f"{module}.lora_B" for module in wrappers\n    }\n    assert set(state) == expected_keys\n    copied = 0\n    with torch.no_grad():\n        for key, value in state.items():\n            target = parameters[key]\n            assert tuple(target.shape) == tuple(value.shape)\n            target.copy_(value.to(device=target.device, dtype=target.dtype))\n            copied += target.numel()\n    assert copied == ADAPTER_PARAMETERS\n    del raw_adapter, state, parameters\n    tokenizer = load_tokenizer(card_name)\n    labels = labels_from_tokenizer(tokenizer)\n    labels_sha256 = hashlib.sha256("\\0".join(labels).encode("utf-8")).hexdigest()\n    assert labels_sha256 == SPEC["expected_labels_sha256"], (\n        labels_sha256,\n        SPEC["expected_labels_sha256"],\n    )\n    pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)\n    model_loaded = True\n    torch.cuda.synchronize()\n    emit(\n        "model_loaded",\n        base_parameters=BASE_PARAMETERS,\n        adapter_parameters=ADAPTER_PARAMETERS,\n        total_parameters=TOTAL_PARAMETERS,\n        lora_wrappers=len(wrappers),\n    )\n\n\ndef set_adapter(enabled: bool) -> None:\n    for wrapper in wrappers.values():\n        wrapper.enabled = bool(enabled)\n    assert all(wrapper.enabled is bool(enabled) for wrapper in wrappers.values())\n\n\ndef capture_one(waveform: np.ndarray, sample_rate: int, omni_code: str) -> np.ndarray:\n    assert pipe is not None and labels is not None\n    captured: list[torch.Tensor] = []\n    original_forward = pipe.model.forward\n\n    def spy(source_seqs: Any, source_seq_lens: Any, *args: Any, **kwargs: Any) -> Any:\n        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)\n        logits, layout = output[0], output[1]\n        length = layout.seq_lens[0]\n        length = int(length.item() if hasattr(length, "item") else length)\n        value = torch.log_softmax(logits[0, :length].detach().float(), dim=-1).cpu()\n        captured.append(value)\n        return output\n\n    pipe.model.forward = spy\n    try:\n        with torch.inference_mode():\n            pipe.transcribe(\n                [{"waveform": waveform, "sample_rate": int(sample_rate)}],\n                lang=[omni_code],\n                batch_size=1,\n            )\n    finally:\n        pipe.model.forward = original_forward\n    assert len(captured) == 1\n    result = captured[0].numpy()\n    assert result.ndim == 2 and result.shape[1] == len(labels)\n    assert np.isfinite(result).all()\n    return result\n\n\n_B64_CHARS = set(b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=\\n\\r")\n\n\ndef ffmpeg_decode(raw: bytes, sample_rate: int = 16_000) -> tuple[np.ndarray | None, int | None]:\n    try:\n        process = subprocess.run(\n            [\n                "ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",\n                "-f", "f32le", "-ac", "1", "-ar", str(sample_rate), "pipe:1",\n            ],\n            input=raw,\n            capture_output=True,\n            check=True,\n        )\n        waveform = np.frombuffer(process.stdout, dtype=np.float32).copy()\n        return (waveform, sample_rate) if len(waveform) else (None, None)\n    except Exception:\n        return None, None\n\n\ndef decode_audio(cell: Any) -> tuple[np.ndarray | None, int | None]:\n    raw = cell.get("bytes") if isinstance(cell, dict) else cell\n    if raw is None:\n        return None, None\n    if hasattr(raw, "tobytes"):\n        raw = raw.tobytes()\n    elif isinstance(raw, (memoryview, bytearray)):\n        raw = bytes(raw)\n    if not isinstance(raw, bytes):\n        return None, None\n    if raw[:5] == b"UklGR" or (len(raw) > 32 and set(raw[:80]) <= _B64_CHARS):\n        try:\n            decoded = base64.b64decode(raw, validate=False)\n            if decoded[:4] in (b"RIFF", b"fLaC", b"OggS") or decoded[:3] == b"ID3":\n                raw = decoded\n        except Exception:\n            pass\n    try:\n        waveform, sample_rate = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)\n        if getattr(waveform, "ndim", 1) > 1:\n            waveform = waveform.mean(axis=1)\n        return np.asarray(waveform, dtype=np.float32), int(sample_rate)\n    except Exception:\n        return ffmpeg_decode(raw)\n\n\nCHUNKING = SPEC["chunking"]\n\n\ndef windows(waveform: np.ndarray, sample_rate: int) -> list[tuple[np.ndarray, bool, bool]]:\n    total = len(waveform)\n    threshold = int(float(CHUNKING["short_threshold_s"]) * sample_rate)\n    if total <= threshold:\n        return [(waveform, False, False)]\n    window = int(float(CHUNKING["window_s"]) * sample_rate)\n    hop = int(\n        (float(CHUNKING["window_s"]) - float(CHUNKING["overlap_s"])) * sample_rate\n    )\n    output = []\n    offset = 0\n    while offset < total:\n        segment = waveform[offset : offset + window]\n        last = offset + window >= total\n        if len(segment) >= int(0.2 * sample_rate):\n            output.append((segment, offset > 0, not last))\n        if last:\n            break\n        offset += hop\n    assert output\n    return output\n\n\n_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")\n_PUNCT_RE = re.compile(r"[^\\w\\s\'’\\-]", re.UNICODE)\n_WS_RE = re.compile(r"\\s+")\n\n\ndef text_badness(value: Any) -> int:\n    value = str(value)\n    score = 50 * value.count("�")\n    score += 6 * sum(character in _MOJIBAKE_MARKERS for character in value)\n    score += 10 * sum(\n        ord(character) < 32 and character not in "\\n\\t\\r" for character in value\n    )\n    return score\n\n\ndef repair_mojibake(value: Any) -> str:\n    raw = str(value or "")\n    candidates = [(raw, "none")]\n    if any(character in _MOJIBAKE_MARKERS for character in raw) or "�" in raw:\n        for encoding in ("latin-1", "cp1252"):\n            try:\n                candidates.append((raw.encode(encoding).decode("utf-8"), encoding))\n            except (UnicodeEncodeError, UnicodeDecodeError):\n                continue\n    best, _ = min(candidates, key=lambda item: text_badness(item[0]))\n    return best if text_badness(best) < text_badness(raw) else raw\n\n\ndef normalize_text(value: str) -> str:\n    value = unicodedata.normalize("NFC", repair_mojibake(value)).lower()\n    value = _PUNCT_RE.sub(" ", value).replace("_", " ")\n    return _WS_RE.sub(" ", value).strip()\n\n\ndef decoder_kwargs(config: dict[str, Any]) -> dict[str, Any]:\n    # 20.K5 a audite uniquement alpha/beta a la construction et beam au decode.\n    # Ne pas introduire ici de token_min_logp/beam_prune_logp non audites.\n    return {"beam_width": int(config["beam"])}\n\n\ndef build_group_decoder(group: str) -> None:\n    global decoder, decoder_group, resident_decoders, max_resident_decoders\n    assert labels is not None\n    if decoder is not None:\n        try:\n            decoder.cleanup()\n        except Exception:\n            pass\n        decoder = None\n        decoder_group = None\n        resident_decoders = 0\n        try:\n            from pyctcdecode.decoder import BeamSearchDecoderCTC\n            BeamSearchDecoderCTC.clear_class_models()\n        except Exception:\n            pass\n        malloc_trim()\n    from pyctcdecode import build_ctcdecoder\n\n    config = RUNTIME_CONFIG[group]\n    binary = Path(config["lm_bin"])\n    unigrams_path = Path(config["uni_file"])\n    assert binary.is_file() and sha256_file(binary) == config["binary_sha256"]\n    assert unigrams_path.is_file() and sha256_file(unigrams_path) == config["unigrams_sha256"]\n    unigrams = unigrams_path.read_text(encoding="utf-8").splitlines()\n    assert unigrams\n    decoder = build_ctcdecoder(\n        labels,\n        str(binary),\n        unigrams=unigrams,\n        alpha=float(config["alpha"]),\n        beta=float(config["beta"]),\n    )\n    decoder_group = group\n    resident_decoders = 1\n    max_resident_decoders = max(max_resident_decoders, resident_decoders)\n    assert resident_decoders == 1\n    emit("decoder_loaded", group=group, asset_id=config["asset_id"])\n\n\ndef transcribe_waveform(\n    waveform: np.ndarray,\n    sample_rate: int,\n    omni_code: str,\n    config: dict[str, Any],\n) -> str:\n    assert decoder is not None\n    if len(waveform) < int(0.2 * sample_rate):\n        waveform = np.pad(waveform, (0, int(0.2 * sample_rate) - len(waveform)))\n    segments = windows(waveform, sample_rate)\n    parts = []\n    for segment, trim_left, trim_right in segments:\n        logits = capture_one(segment, sample_rate, omni_code)\n        frames_per_second = len(logits) / (len(segment) / float(sample_rate))\n        trim = int(round(float(CHUNKING["trim_s"]) * frames_per_second))\n        left = trim if trim_left else 0\n        right = len(logits) - (trim if trim_right else 0)\n        assert right > left, (len(logits), left, right)\n        parts.append(logits[left:right])\n    full = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)\n    prediction = decoder.decode(full, **decoder_kwargs(config))\n    return normalize_text(prediction)\n\n\ndef stage_shard(record: dict[str, Any]) -> tuple[Path, str]:\n    source = Path(record["source_path"]).resolve()\n    assert source.is_file() and source.stat().st_size == int(record["bytes"])\n    assert "kaggle_test_full" in str(source).lower(), source\n    token = hashlib.sha256(record["relative_path"].encode()).hexdigest()[:16]\n    target = LOCAL_STAGE_ROOT / f"{token}_{source.name}"\n    partial = target.with_suffix(target.suffix + ".partial")\n    if target.is_file() and target.stat().st_size != int(record["bytes"]):\n        target.unlink()\n    if not target.is_file():\n        subprocess.run(\n            ["rsync", "-a", "--partial", "--append-verify", str(source), str(partial)],\n            check=True,\n        )\n        assert partial.stat().st_size == int(record["bytes"])\n        os.replace(partial, target)\n    assert target.stat().st_size == int(record["bytes"])\n    source_sha = sha256_file(target)\n    if source_sha != record["source_sha256"]:\n        target.unlink()\n        try:\n            partial.unlink()\n        except FileNotFoundError:\n            pass\n        subprocess.run(\n            ["rsync", "-a", "--partial", "--append-verify", str(source), str(partial)],\n            check=True,\n        )\n        assert partial.stat().st_size == int(record["bytes"])\n        os.replace(partial, target)\n        source_sha = sha256_file(target)\n    assert source_sha == record["source_sha256"], record["relative_path"]\n    assert pq.ParquetFile(target).metadata.num_rows == int(record["rows"])\n    return target, source_sha\n\n\ndef load_partial_cache(\n    record: dict[str, Any], source_sha: str\n) -> tuple[list[str], list[str], int, int]:\n    partial, meta_path = partial_cache_paths(record)\n    if not partial.is_file() or not meta_path.is_file():\n        return [], [], 0, 0\n    try:\n        meta = json.loads(meta_path.read_text(encoding="utf-8"))\n        assert meta["status"] == "PARTIAL_SHARD_CACHE"\n        assert meta["run_id"] == RUN_ID\n        assert meta["contract_sha256"] == CONTRACT_SHA256\n        assert meta["base_spec_sha256"] == canonical_sha256(base_shard_spec(record))\n        assert meta["source_sha256"] == source_sha\n        assert meta["partial_sha256"] == sha256_file(partial)\n        frame = pd.read_parquet(partial)\n        assert list(frame.columns) == ["id", "language", "transcription"]\n        ids = frame["id"].astype(str).tolist()\n        predictions = frame["transcription"].fillna("").astype(str).tolist()\n        assert 0 < len(frame) < int(record["rows"])\n        assert len(ids) == len(set(ids))\n        assert sequence_digest(ids) == meta["ids_sha256"]\n        assert frame["language"].astype(str).eq(record["submission_language"]).all()\n        return (\n            ids,\n            predictions,\n            int(meta.get("decode_errors", 0)),\n            int(meta.get("segmented_clips", 0)),\n        )\n    except Exception:\n        for path in (partial, meta_path):\n            try:\n                path.unlink()\n            except FileNotFoundError:\n                pass\n        return [], [], 0, 0\n\n\ndef save_partial_cache(\n    record: dict[str, Any],\n    source_sha: str,\n    ids: list[str],\n    predictions: list[str],\n    decode_errors: int,\n    segmented_clips: int,\n) -> None:\n    partial, meta_path = partial_cache_paths(record)\n    assert len(ids) == len(predictions)\n    assert 0 < len(ids) < int(record["rows"])\n    assert len(ids) == len(set(ids))\n    frame = pd.DataFrame(\n        {\n            "id": ids,\n            "language": record["submission_language"],\n            "transcription": predictions,\n        }\n    )\n    atomic_parquet(frame, partial)\n    atomic_json(\n        {\n            "schema": 1,\n            "status": "PARTIAL_SHARD_CACHE",\n            "run_id": RUN_ID,\n            "contract_sha256": CONTRACT_SHA256,\n            "base_spec_sha256": canonical_sha256(base_shard_spec(record)),\n            "source_sha256": source_sha,\n            "partial_sha256": sha256_file(partial),\n            "rows": len(frame),\n            "ids_sha256": sequence_digest(ids),\n            "decode_errors": int(decode_errors),\n            "segmented_clips": int(segmented_clips),\n        },\n        meta_path,\n    )\n\n\nmissing_by_group: OrderedDict[str, list[dict[str, Any]]] = OrderedDict()\nfor record in missing:\n    missing_by_group.setdefault(record["group"], []).append(record)\n\nprocessed_new_clips = 0\ndecode_errors = cached_decode_errors\nnew_total = sum(int(row["rows"]) for row in missing)\nshards_completed_new = 0\n\nif missing:\n    load_runtime()\n\nfor group, group_records in missing_by_group.items():\n    assert group in set(SPEC["active_test_groups"])\n    set_adapter(group == "mas|unscripted")\n    build_group_decoder(group)\n    config = RUNTIME_CONFIG[group]\n    omni_code = str(group_records[0]["omni_language"])\n    for record in group_records:\n        output, meta_path = cache_paths(record)\n        local_shard, source_sha = stage_shard(record)\n        base_spec = base_shard_spec(record)\n        base_spec_sha = canonical_sha256(base_spec)\n        ids, predictions, shard_decode_errors, segmented_clips = load_partial_cache(\n            record, source_sha\n        )\n        decode_errors += shard_decode_errors\n        assert decode_errors <= int(SPEC["max_decode_errors"]), (\n            decode_errors,\n            SPEC["max_decode_errors"],\n        )\n        resumed_rows = len(ids)\n        if resumed_rows:\n            processed_new_clips += resumed_rows\n            emit(\n                "partial_cache_reused",\n                group=group,\n                shard=record["relative_path"],\n                rows=resumed_rows,\n            )\n        shard_started = time.monotonic()\n        parquet_file = pq.ParquetFile(local_shard)\n        source_row_index = 0\n        for batch in parquet_file.iter_batches(batch_size=8, columns=["id", "audio"]):\n            for row in batch.to_pylist():\n                if source_row_index < resumed_rows:\n                    assert str(row["id"]) == ids[source_row_index]\n                    source_row_index += 1\n                    continue\n                identifier = str(row["id"])\n                waveform, sample_rate = decode_audio(row["audio"])\n                if waveform is None or sample_rate is None or len(waveform) == 0:\n                    prediction = ""\n                    prospective_errors = decode_errors + 1\n                    emit(\n                        "audio_decode_error",\n                        group=group,\n                        id_sha256=hashlib.sha256(identifier.encode()).hexdigest(),\n                        audio_decode_errors=prospective_errors,\n                    )\n                    if prospective_errors > int(SPEC["max_decode_errors"]):\n                        if ids:\n                            save_partial_cache(\n                                record,\n                                source_sha,\n                                ids,\n                                predictions,\n                                shard_decode_errors,\n                                segmented_clips,\n                            )\n                        raise RuntimeError(\n                            f"Trop d\'erreurs audio : {prospective_errors} > "\n                            f"{SPEC[\'max_decode_errors\']}"\n                        )\n                    shard_decode_errors += 1\n                    decode_errors = prospective_errors\n                else:\n                    was_segmented = len(waveform) > int(\n                        float(CHUNKING["short_threshold_s"]) * int(sample_rate)\n                    )\n                    for attempt in (1, 2):\n                        try:\n                            prediction = transcribe_waveform(\n                                waveform, int(sample_rate), omni_code, config\n                            )\n                            break\n                        except Exception as error:\n                            emit(\n                                "inference_retry" if attempt == 1 else "inference_fatal",\n                                group=group,\n                                attempt=attempt,\n                                error_type=type(error).__name__,\n                                id_sha256=hashlib.sha256(identifier.encode()).hexdigest(),\n                            )\n                            torch.cuda.empty_cache()\n                            malloc_trim()\n                            if attempt == 2:\n                                if ids:\n                                    save_partial_cache(\n                                        record,\n                                        source_sha,\n                                        ids,\n                                        predictions,\n                                        shard_decode_errors,\n                                        segmented_clips,\n                                    )\n                                raise RuntimeError(\n                                    "Erreur d\'inference persistante ; reprise sure au dernier "\n                                    "point partiel, sans mise en cache de l\'ID fautif."\n                                ) from error\n                    if was_segmented:\n                        segmented_clips += 1\n                ids.append(identifier)\n                predictions.append(prediction)\n                processed_new_clips += 1\n                source_row_index += 1\n                if (\n                    len(ids) % int(SPEC["partial_flush_every"]) == 0\n                    and len(ids) < int(record["rows"])\n                ):\n                    save_partial_cache(\n                        record,\n                        source_sha,\n                        ids,\n                        predictions,\n                        shard_decode_errors,\n                        segmented_clips,\n                    )\n                if processed_new_clips % 100 == 0 or processed_new_clips == new_total:\n                    elapsed = time.monotonic() - STARTED\n                    rate = processed_new_clips / max(elapsed, 1e-9)\n                    remaining = new_total - processed_new_clips\n                    emit(\n                        "progress",\n                        group=group,\n                        new_clips_done=processed_new_clips,\n                        new_clips_total=new_total,\n                        clips_per_second=rate,\n                        eta_hours=remaining / max(rate, 1e-9) / 3600,\n                    )\n        assert len(ids) == int(record["rows"])\n        assert len(ids) == len(set(ids))\n        assert sequence_digest(ids) == record["ids_sha256"]\n        frame = pd.DataFrame(\n            {\n                "id": ids,\n                "language": record["submission_language"],\n                "transcription": predictions,\n            }\n        )\n        atomic_parquet(frame, output)\n        cache_sha = sha256_file(output)\n        meta = {\n            "schema": 1,\n            "status": "PASS_SHARD_CACHE",\n            "run_id": RUN_ID,\n            "contract_sha256": CONTRACT_SHA256,\n            "base_spec": base_spec,\n            "base_spec_sha256": base_spec_sha,\n            "source_sha256": source_sha,\n            "cache_sha256": cache_sha,\n            "rows": len(frame),\n            "empty_predictions": int(frame["transcription"].astype(str).str.strip().eq("").sum()),\n            "decode_errors": int(shard_decode_errors),\n            "segmented_clips": int(segmented_clips),\n            "partial_rows_reused": int(resumed_rows),\n            "seconds": time.monotonic() - shard_started,\n            "batch_size": 1,\n            "adapter_enabled": group == "mas|unscripted",\n        }\n        atomic_json(meta, meta_path)\n        valid, checked = valid_cache(record)\n        assert valid and checked is not None\n        for partial_path in partial_cache_paths(record):\n            try:\n                partial_path.unlink()\n            except FileNotFoundError:\n                pass\n        cache_records.append(\n            {\n                "relative_path": record["relative_path"],\n                "output": str(output),\n                "meta": str(meta_path),\n                "cache_sha256": cache_sha,\n                "source_sha256": source_sha,\n                "rows": len(frame),\n                "reused": False,\n            }\n        )\n        shards_completed_new += 1\n        emit(\n            "shard_complete",\n            group=group,\n            shard=record["relative_path"],\n            rows=len(frame),\n            shards_completed_new=shards_completed_new,\n            shards_missing_total=len(missing),\n        )\n        del frame, predictions, ids, parquet_file\n        try:\n            local_shard.unlink()\n        except FileNotFoundError:\n            pass\n        malloc_trim()\n\nif decoder is not None:\n    try:\n        decoder.cleanup()\n    except Exception:\n        pass\n    decoder = None\n    resident_decoders = 0\n    try:\n        from pyctcdecode.decoder import BeamSearchDecoderCTC\n        BeamSearchDecoderCTC.clear_class_models()\n    except Exception:\n        pass\n    malloc_trim()\n\n# Validation finale de tous les caches, y compris ceux repris.\nfinal_records = []\nfor record in TEST_RECORDS:\n    valid, meta = valid_cache(record)\n    assert valid and meta is not None, record["relative_path"]\n    output, meta_path = cache_paths(record)\n    final_records.append(\n        {\n            "relative_path": record["relative_path"],\n            "output": str(output),\n            "meta": str(meta_path),\n            "cache_sha256": meta["cache_sha256"],\n            "source_sha256": meta["source_sha256"],\n            "rows": int(record["rows"]),\n            "empty_predictions": int(meta["empty_predictions"]),\n            "decode_errors": int(meta.get("decode_errors", 0)),\n            "segmented_clips": int(meta.get("segmented_clips", 0)),\n        }\n    )\n\nRESULT = {\n    "schema": 1,\n    "cell": "20.K6-worker",\n    "status": "PASS_CACHE_COMPLETE",\n    "run_id": RUN_ID,\n    "contract_sha256": CONTRACT_SHA256,\n    "shards": len(final_records),\n    "clips": sum(row["rows"] for row in final_records),\n    "shards_recomputed": len(missing),\n    "shards_reused": len(TEST_RECORDS) - len(missing),\n    "clips_recomputed": processed_new_clips,\n    "clips_reused": cached_clips,\n    "audio_decode_errors_new": decode_errors - cached_decode_errors,\n    "audio_decode_errors_total": sum(row["decode_errors"] for row in final_records),\n    # Alias conserves pour la compatibilite avec les rapports 17.2/K6.\n    "decode_errors_new": decode_errors - cached_decode_errors,\n    "decode_errors_total": sum(row["decode_errors"] for row in final_records),\n    "empty_predictions_total": sum(row["empty_predictions"] for row in final_records),\n    "segmented_clips_total": sum(row["segmented_clips"] for row in final_records),\n    "model_loaded": model_loaded,\n    "labels_sha256": labels_sha256,\n    "batch_size": 1,\n    "max_resident_decoders": max_resident_decoders,\n    "active_test_groups": list(SPEC["active_test_groups"]),\n    "gpu": torch.cuda.get_device_name(0),\n    "gpu_peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),\n    "gpu_peak_reserved_bytes": int(torch.cuda.max_memory_reserved()),\n    "rss_bytes": int(psutil.Process(os.getpid()).memory_info().rss),\n    "elapsed_seconds": time.monotonic() - STARTED,\n    "cache_records": final_records,\n    "raw_audio_written_to_drive": False,\n    "logits_written": False,\n    "kaggle_submission_performed": False,\n}\natomic_json(RESULT, RESULT_PATH)\nemit("worker_complete", shards=94, clips=41_733)\n'

"""20.K6 - inference test finale avec le package exact valide par 20.K5.

La cellule ne retune rien, ne modifie pas la configuration active et ne soumet
rien a Kaggle. Elle inventorie le test, lance un worker GPU reprenable, assemble
les 94 caches signes, puis publie un CSV uniquement apres une QA bloquante.
"""

import hashlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from collections import Counter, OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import pyarrow.parquet as pq
import torch


# ---------------------------------------------------------------------------
# 1. Contrat immuable de 20.K5 et attentes Kaggle
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/afrivoices_project")
FT_ROOT = PROJECT_ROOT / "finetune_runs/experiment_run"
REPORT_ROOT = FT_ROOT / "reports"
K5_ROOT = REPORT_ROOT / "kenlm_v6_20_K5"
K6_ROOT = REPORT_ROOT / "kenlm_v6_20_K6"
K6_CACHE_ROOT = REPORT_ROOT / "test_predictions_k6"
TEST_ROOT = PROJECT_ROOT / "kaggle_test_full"
ACTIVE_CONFIG = REPORT_ROOT / "kenlm_tuning_by_domain_v3.json"

EXPECTED_K5_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K5_CONTRACT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K5_REPORT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K5_COMPLETE_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K5_STATUS = "PASS_READY_FOR_20_K6_TEST_INFERENCE"

EXPECTED_ROWS = 41_733
EXPECTED_SHARDS = 94
EXPECTED_LANGUAGE_COUNTS = OrderedDict(
    (("kik", 9192), ("kln", 4837), ("luo", 7437),
     ("mas", 3789), ("som", 3925), ("swa", 12553))
)
SUBMISSION_TO_MANIFEST = {
    "swa": "sw", "kik": "kik", "kln": "kln",
    "luo": "luo", "som": "som", "mas": "mas",
}
SUBMISSION_TO_OMNI = {
    "swa": "swh_Latn", "kik": "kik_Latn", "kln": "kln_Latn",
    "luo": "luo_Latn", "som": "som_Latn", "mas": "saq_Latn",
}
RUNTIME_LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
RUNTIME_GROUPS = tuple(
    f"{language}|{domain}"
    for language in RUNTIME_LANGUAGES
    for domain in ("scripted", "unscripted")
)
ACTIVE_TEST_GROUPS = (
    "kik|scripted", "kik|unscripted",
    "kln|scripted", "kln|unscripted",
    "luo|scripted", "luo|unscripted",
    "mas|scripted", "mas|unscripted",
    "som|scripted", "som|unscripted",
    "sw|unscripted",
)
EXPECTED_GROUP_SHARDS = OrderedDict(
    (
        ("kik|scripted", 5), ("kik|unscripted", 13),
        ("kln|scripted", 2), ("kln|unscripted", 9),
        ("luo|scripted", 5), ("luo|unscripted", 12),
        ("mas|scripted", 2), ("mas|unscripted", 9),
        ("som|scripted", 2), ("som|unscripted", 9),
        ("sw|unscripted", 26),
    )
)
assert sum(EXPECTED_GROUP_SHARDS.values()) == EXPECTED_SHARDS

BASE_PARAMETERS = 975_675_056
ADAPTER_PARAMETERS = 9_898_496
TOTAL_PARAMETERS = 985_573_552
LORA_PLAN_ROWS = 289
BATCH_SIZE = 1
PARTIAL_FLUSH_EVERY = 50
MAX_DECODE_ERRORS = 50
MAX_EMPTY_PREDICTIONS = 50
CHUNKING = {
    "short_threshold_s": 39.5,
    "window_s": 38.0,
    "overlap_s": 6.0,
    "trim_s": 3.0,
}

assert Path("/content/persistent_storage").is_dir(), "Monter Google Drive avant 20.K6."
assert "K6_WORKER_SOURCE" in globals(), "Le worker embarque 20.K6 est absent."
assert torch.cuda.is_available(), "Choisir un runtime GPU L4 ou A100."
assert torch.cuda.is_bf16_supported(), "Le GPU doit prendre BF16 en charge."
assert TOTAL_PARAMETERS == BASE_PARAMETERS + ADAPTER_PARAMETERS < 1_000_000_000
assert BATCH_SIZE == 1


# ---------------------------------------------------------------------------
# 2. Utilitaires
# ---------------------------------------------------------------------------

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path | str, block_size: int = 16 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_json(value: Any) -> str:
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    )


def canonical_sha256(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def sequence_digest(values: list[str]) -> str:
    digest = hashlib.sha256()
    for value in values:
        encoded = str(value).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def read_json(path: Path | str) -> dict[str, Any]:
    with open(path, encoding="utf-8") as handle:
        value = json.load(handle)
    assert isinstance(value, dict), path
    return value


def atomic_json(value: Any, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(frame: pd.DataFrame, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False, lineterminator="\n")
    os.replace(temporary, path)
    print("csv  ->", path)


def require_child(path: Path | str, root: Path | str) -> Path:
    path = Path(path).resolve()
    root = Path(root).resolve()
    assert path == root or root in path.parents, (path, root)
    return path


def artifact_metadata(path: Path | str) -> dict[str, Any]:
    path = Path(path)
    return {"bytes": path.stat().st_size, "sha256": sha256_file(path)}


def validate_artifacts(root: Path, artifacts: dict[str, Any]) -> None:
    for relative, metadata in artifacts.items():
        path = require_child(root / relative, root)
        assert path.is_file(), path
        assert path.stat().st_size == int(metadata["bytes"]), (path, "size")
        assert sha256_file(path) == metadata["sha256"], (path, "sha256")


def copy_atomic_verified(source: Path, target: Path, expected_sha256: str) -> None:
    assert source.is_file(), source
    assert sha256_file(source) == expected_sha256, source
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and sha256_file(target) == expected_sha256:
        return
    temporary = target.with_name(target.name + f".tmp-{os.getpid()}")
    shutil.copy2(source, temporary)
    assert sha256_file(temporary) == expected_sha256
    os.replace(temporary, target)


def package_versions() -> dict[str, str]:
    result = {}
    for name in (
        "torch", "fairseq2", "omnilingual-asr", "numpy", "pandas", "pyarrow",
        "soundfile", "pyctcdecode", "kenlm", "psutil",
    ):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = "UNKNOWN"
    return result


def git_head(path: Path) -> str:
    result = subprocess.run(
        ["git", "-C", str(path), "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    assert re.fullmatch(r"[0-9a-f]{40}", result)
    return result


def canonical_domain(value: str) -> str:
    value = str(value).strip().lower().replace("_", "-")
    if "unscript" in value or "spont" in value:
        return "unscripted"
    if "script" in value or "read" in value:
        return "scripted"
    raise AssertionError(f"Domaine test inconnu : {value!r}")


# Contrat de normalisation WER exact de K4/0.36878.
_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")
_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_WS_RE = re.compile(r"\s+")


def text_badness(value: Any) -> int:
    value = str(value)
    score = 50 * value.count("�")
    score += 6 * sum(character in _MOJIBAKE_MARKERS for character in value)
    score += 10 * sum(
        ord(character) < 32 and character not in "\n\t\r" for character in value
    )
    return score


def repair_mojibake(value: Any) -> str:
    raw = str(value or "")
    candidates = [(raw, "none")]
    if any(character in _MOJIBAKE_MARKERS for character in raw) or "�" in raw:
        for encoding in ("latin-1", "cp1252"):
            try:
                candidates.append((raw.encode(encoding).decode("utf-8"), encoding))
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
    best, _ = min(candidates, key=lambda item: text_badness(item[0]))
    return best if text_badness(best) < text_badness(raw) else raw


def normalize_wer_text(value: Any) -> str:
    value = unicodedata.normalize("NFC", repair_mojibake(value)).lower()
    value = _PUNCT_RE.sub(" ", value).replace("_", " ")
    return _WS_RE.sub(" ", value).strip()


NORMALIZER_SPEC = {
    "name": "normalize_wer_text_k4_v1",
    "unicode": "NFC",
    "case": "lower",
    "mojibake_candidates": ["identity", "latin-1_to_utf8", "cp1252_to_utf8"],
    "punctuation_regex": r"[^\w\s'’\-]",
    "underscore": "space",
    "whitespace": "collapse_strip",
}


# ---------------------------------------------------------------------------
# 3. Validation cryptographique de 20.K5 et de son package
# ---------------------------------------------------------------------------

LATEST_PATH = K5_ROOT / "LATEST_PASS.json"
K5_LATEST = read_json(LATEST_PATH)
assert K5_LATEST["cell"] == "20.K5"
assert K5_LATEST["status"] == EXPECTED_K5_STATUS
assert K5_LATEST["run_id"] == EXPECTED_K5_RUN_ID
assert K5_LATEST["contract_sha256"] == EXPECTED_K5_CONTRACT_SHA256
assert K5_LATEST["report_sha256"] == EXPECTED_K5_REPORT_SHA256
assert K5_LATEST["complete_sha256"] == EXPECTED_K5_COMPLETE_SHA256
assert K5_LATEST.get("active_runtime_modified") is False
assert K5_LATEST.get("submission_created") is False

K5_REPORT_PATH = require_child(K5_LATEST["report"], K5_ROOT)
K5_COMPLETE_PATH = require_child(K5_LATEST["complete"], K5_ROOT)
PACKAGE_ROOT = require_child(K5_LATEST["package"], FT_ROOT)
assert sha256_file(K5_REPORT_PATH) == EXPECTED_K5_REPORT_SHA256
assert sha256_file(K5_COMPLETE_PATH) == EXPECTED_K5_COMPLETE_SHA256

K5_REPORT = read_json(K5_REPORT_PATH)
K5_COMPLETE = read_json(K5_COMPLETE_PATH)
K5_RUN_ROOT = K5_REPORT_PATH.parent
assert K5_REPORT["status"] == EXPECTED_K5_STATUS
assert K5_REPORT["run_id"] == EXPECTED_K5_RUN_ID
assert K5_REPORT["contract_sha256"] == EXPECTED_K5_CONTRACT_SHA256
assert int(K5_REPORT["neural_parameters"]) == TOTAL_PARAMETERS
assert K5_REPORT.get("test_audio_read") is False
assert K5_REPORT.get("active_runtime_modified") is False
assert K5_REPORT.get("submission_created") is False
assert K5_COMPLETE["status"] == EXPECTED_K5_STATUS
assert K5_COMPLETE["run_id"] == EXPECTED_K5_RUN_ID
assert K5_COMPLETE["contract_sha256"] == EXPECTED_K5_CONTRACT_SHA256
validate_artifacts(K5_RUN_ROOT, K5_COMPLETE["artifacts"])
EXPECTED_LABELS_SHA256 = K5_REPORT["worker"]["labels_sha256"]
assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_LABELS_SHA256)

PACKAGE_MANIFEST_PATH = PACKAGE_ROOT / "package_manifest.json"
PACKAGE_MANIFEST = read_json(PACKAGE_MANIFEST_PATH)
assert sha256_file(PACKAGE_MANIFEST_PATH) == K5_REPORT["package_manifest_sha256"]
assert PACKAGE_MANIFEST["run_id"] == EXPECTED_K5_RUN_ID
assert PACKAGE_MANIFEST["contract_sha256"] == EXPECTED_K5_CONTRACT_SHA256
assert PACKAGE_MANIFEST["status"] == "CANDIDATE_NOT_ACTIVE"
assert PACKAGE_MANIFEST["path_mode"] == "relative_to_package_root"
assert tuple(PACKAGE_MANIFEST["runtime_groups"]) == RUNTIME_GROUPS
validate_artifacts(PACKAGE_ROOT, PACKAGE_MANIFEST["artifacts"])

RUNTIME_CONFIG_PATH = PACKAGE_ROOT / "runtime_config.json"
RUNTIME_CONFIG = read_json(RUNTIME_CONFIG_PATH)
assert set(RUNTIME_CONFIG) == set(RUNTIME_GROUPS)
for group, config in RUNTIME_CONFIG.items():
    binary = require_child(PACKAGE_ROOT / config["lm_bin"], PACKAGE_ROOT)
    unigrams = require_child(PACKAGE_ROOT / config["uni_file"], PACKAGE_ROOT)
    assert binary.is_file() and sha256_file(binary) == config["binary_sha256"]
    assert unigrams.is_file() and sha256_file(unigrams) == config["unigrams_sha256"]
    assert int(config["beam"]) in (100, 250)

EXTERNAL = PACKAGE_MANIFEST["external_immutable_assets"]
assert set(EXTERNAL) == {"base_weight", "adapter_weight", "lora_plan"}
for role, proof in EXTERNAL.items():
    path = require_child(proof["path"], PROJECT_ROOT)
    assert path.is_file() and sha256_file(path) == proof["sha256"], role

BASE_WEIGHT = Path(EXTERNAL["base_weight"]["path"]).resolve()
ADAPTER_WEIGHT = Path(EXTERNAL["adapter_weight"]["path"]).resolve()
LORA_PLAN = Path(EXTERNAL["lora_plan"]["path"]).resolve()
BASE_SHA = EXTERNAL["base_weight"]["sha256"]
ADAPTER_SHA = EXTERNAL["adapter_weight"]["sha256"]
PLAN_SHA = EXTERNAL["lora_plan"]["sha256"]
plan = pd.read_parquet(LORA_PLAN)
assert len(plan) == LORA_PLAN_ROWS
assert int(plan["parameters_per_adapter"].sum()) == ADAPTER_PARAMETERS

assert ACTIVE_CONFIG.is_file(), ACTIVE_CONFIG
ACTIVE_CONFIG_SHA_BEFORE = sha256_file(ACTIVE_CONFIG)
assert ACTIVE_CONFIG_SHA_BEFORE == read_json(K5_RUN_ROOT / "contract.json")[
    "active_config_sha256_before"
], "La configuration active a change depuis 20.K5."

print("=== PREFLIGHT 20.K6 ===")
print("20.K5 exact  :", EXPECTED_K5_RUN_ID)
print("Package      :", PACKAGE_ROOT)
print("Base BF16    :", BASE_SHA[:16])
print("LoRA Maasai  :", ADAPTER_SHA[:16], "| uniquement mas|unscripted")
print("Parametres   :", f"{TOTAL_PARAMETERS:,}")
print("GPU          :", torch.cuda.get_device_name(0), "| BF16=", torch.cuda.is_bf16_supported())


# ---------------------------------------------------------------------------
# 4. Inventaire fort des 94 shards test (aucun audio decode ici)
# ---------------------------------------------------------------------------

assert TEST_ROOT.is_dir(), TEST_ROOT
language_order = {language: index for index, language in enumerate(EXPECTED_LANGUAGE_COUNTS)}
paths = sorted(
    TEST_ROOT.rglob("*.parquet"),
    key=lambda path: (
        language_order.get(path.relative_to(TEST_ROOT).parts[0], 999),
        str(path.relative_to(TEST_ROOT)).lower(),
    ),
)
assert len(paths) == EXPECTED_SHARDS, (len(paths), EXPECTED_SHARDS)

test_records = []
all_ids: list[str] = []
test_counts = Counter()
print("\nInventaire SHA256 complet du test (une seule lecture Drive)...")
for index, path in enumerate(paths, 1):
    relative = path.relative_to(TEST_ROOT)
    assert len(relative.parts) >= 3, relative
    submission_language = relative.parts[0]
    assert submission_language in EXPECTED_LANGUAGE_COUNTS, relative
    domain = canonical_domain(relative.parts[-2])
    manifest_language = SUBMISSION_TO_MANIFEST[submission_language]
    group = f"{manifest_language}|{domain}"
    assert group in ACTIVE_TEST_GROUPS, group
    parquet_file = pq.ParquetFile(path)
    assert {"id", "audio"} <= set(parquet_file.schema_arrow.names), path
    rows = int(parquet_file.metadata.num_rows)
    identifiers = (
        pq.read_table(path, columns=["id"])["id"].to_pandas().astype(str).tolist()
    )
    assert len(identifiers) == rows
    assert len(identifiers) == len(set(identifiers)), relative
    source_sha = sha256_file(path)
    record = {
        "relative_path": str(relative),
        "source_path": str(path.resolve()),
        "submission_language": submission_language,
        "manifest_language": manifest_language,
        "omni_language": SUBMISSION_TO_OMNI[submission_language],
        "domain": domain,
        "group": group,
        "rows": rows,
        "bytes": int(path.stat().st_size),
        "source_sha256": source_sha,
        "ids_sha256": sequence_digest(identifiers),
    }
    test_records.append(record)
    all_ids.extend(identifiers)
    test_counts[submission_language] += rows
    print(
        f"  [{index:02d}/{EXPECTED_SHARDS}] {relative} | {rows} clips | "
        f"{source_sha[:12]}"
    )

assert len(all_ids) == EXPECTED_ROWS
assert len(all_ids) == len(set(all_ids)), "IDs test dupliques entre shards."
assert OrderedDict((key, test_counts[key]) for key in EXPECTED_LANGUAGE_COUNTS) == (
    EXPECTED_LANGUAGE_COUNTS
)
assert set(record["group"] for record in test_records) == set(ACTIVE_TEST_GROUPS)
assert Counter(record["group"] for record in test_records) == Counter(
    EXPECTED_GROUP_SHARDS
), Counter(record["group"] for record in test_records)

portable_inventory = [
    {key: value for key, value in record.items() if key != "source_path"}
    for record in test_records
]
TEST_INVENTORY_SHA = canonical_sha256(portable_inventory)
TEST_ID_SEQUENCE_SHA = sequence_digest(all_ids)


# ---------------------------------------------------------------------------
# 5. Contrat K6 lie au GPU, au worker, au test et a tous les actifs K5
# ---------------------------------------------------------------------------

REPO = Path("/content/omnilingual-asr-k6")
assert (REPO / ".git").is_dir(), "Rejouer 20.K6.A."
OMNI_COMMIT = git_head(REPO)
K5_CONTRACT_DOC = read_json(K5_RUN_ROOT / "contract.json")
K5_SOFTWARE = K5_CONTRACT_DOC["software_fingerprint"]
assert OMNI_COMMIT == K5_SOFTWARE["omnilingual_asr_git_commit"]

CURRENT_PACKAGES = package_versions()
for package in (
    "torch", "fairseq2", "omnilingual-asr", "numpy", "pandas", "pyarrow",
    "soundfile", "pyctcdecode", "kenlm", "psutil",
):
    expected = K5_SOFTWARE["packages"].get(package)
    if expected and expected != "UNKNOWN":
        observed = CURRENT_PACKAGES[package]
        versions_match = (
            observed.split("+")[0] == expected.split("+")[0]
            if package == "torch"
            else observed == expected
        )
        assert versions_match, (
            f"Version {package} differente de 20.K5 : "
            f"{observed} != {expected}. Rejouer 20.K6.A."
        )

SOFTWARE = {
    "python": sys.version.split()[0],
    "packages": CURRENT_PACKAGES,
    "omnilingual_asr_git_commit": OMNI_COMMIT,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
    "bf16": bool(torch.cuda.is_bf16_supported()),
}
ROUTE_CONTRACT = {
    group: {
        "asset_id": config["asset_id"],
        "binary_sha256": config["binary_sha256"],
        "unigrams_sha256": config["unigrams_sha256"],
        "alpha": float(config["alpha"]),
        "beta": float(config["beta"]),
        "beam": int(config["beam"]),
        "decode_kwargs_applied": ["beam_width"],
        "adapter_enabled": group == "mas|unscripted",
    }
    for group, config in RUNTIME_CONFIG.items()
}
CONTRACT = {
    "schema": 1,
    "cell": "20.K6",
    "purpose": "exact_k5_package_full_test_inference_no_retuning",
    "k5": {
        "run_id": EXPECTED_K5_RUN_ID,
        "contract_sha256": EXPECTED_K5_CONTRACT_SHA256,
        "report_sha256": EXPECTED_K5_REPORT_SHA256,
        "complete_sha256": EXPECTED_K5_COMPLETE_SHA256,
        "package_manifest_sha256": sha256_file(PACKAGE_MANIFEST_PATH),
        "runtime_config_sha256": sha256_file(RUNTIME_CONFIG_PATH),
    },
    "neural": {
        "base_parameters": BASE_PARAMETERS,
        "adapter_parameters": ADAPTER_PARAMETERS,
        "total_parameters": TOTAL_PARAMETERS,
        "base_weight_sha256": BASE_SHA,
        "adapter_weight_sha256": ADAPTER_SHA,
        "lora_plan_sha256": PLAN_SHA,
        "lora_plan_rows": LORA_PLAN_ROWS,
        "adapter_route": "mas|unscripted_only",
    },
    "runtime": {
        "device": "cuda",
        "dtype": "bfloat16",
        "batch_size": BATCH_SIZE,
        "tf32": False,
        "max_resident_decoders": 1,
        "routes": ROUTE_CONTRACT,
        "active_test_groups": list(ACTIVE_TEST_GROUPS),
        "expected_labels_sha256": EXPECTED_LABELS_SHA256,
        "chunking": CHUNKING,
        "normalizer": NORMALIZER_SPEC,
        "normalizer_sha256": canonical_sha256(NORMALIZER_SPEC),
    },
    "test": {
        "shards": EXPECTED_SHARDS,
        "rows": EXPECTED_ROWS,
        "language_counts": EXPECTED_LANGUAGE_COUNTS,
        "group_shard_counts": EXPECTED_GROUP_SHARDS,
        "portable_inventory_sha256": TEST_INVENTORY_SHA,
        "id_sequence_sha256": TEST_ID_SEQUENCE_SHA,
    },
    "recovery": {
        "cache_scope": "one_contract_one_gpu_fingerprint",
        "shard_atomic": True,
        "partial_flush_every": PARTIAL_FLUSH_EVERY,
        "raw_audio_or_logits_on_drive": False,
    },
    "qa": {
        "columns": ["id", "language", "transcription"],
        "maximum_decode_errors": MAX_DECODE_ERRORS,
        "maximum_empty_predictions_before_placeholder": MAX_EMPTY_PREDICTIONS,
        "empty_placeholder": "a",
        "maximum_words_per_prediction": 500,
        "no_manual_correction": True,
    },
    "software": SOFTWARE,
    "worker_sha256": hashlib.sha256(K6_WORKER_SOURCE.encode("utf-8")).hexdigest(),
    "active_config_sha256_before": ACTIVE_CONFIG_SHA_BEFORE,
    "kaggle_submission_performed": False,
}
CONTRACT_SHA = canonical_sha256(CONTRACT)
RUN_ID = CONTRACT_SHA[:16]
REPORT_STAGING = K6_ROOT / f"{RUN_ID}.staging"
REPORT_FINAL = K6_ROOT / RUN_ID
CACHE_ROOT = K6_CACHE_ROOT / RUN_ID

print("\nContrat 20.K6 :", CONTRACT_SHA)
print("Run ID         :", RUN_ID)
print("Shards/clips   :", EXPECTED_SHARDS, "/", EXPECTED_ROWS)
print("Reprise        : shard atomique + point partiel toutes les 50 lignes")


# ---------------------------------------------------------------------------
# 6. Reprise d'un PASS existant, sinon preparation et worker GPU
# ---------------------------------------------------------------------------

def validate_completed(root: Path) -> dict[str, Any] | None:
    complete_path = root / "_COMPLETE.json"
    report_path = root / "inference_report_20_K6.json"
    submission_path = root / f"submission_k6_{RUN_ID}.csv"
    if not complete_path.is_file() or not report_path.is_file() or not submission_path.is_file():
        return None
    try:
        complete = read_json(complete_path)
        report = read_json(report_path)
        assert complete["status"] == "PASS_SUBMISSION_READY_FOR_KAGGLE"
        assert complete["run_id"] == RUN_ID
        assert complete["contract_sha256"] == CONTRACT_SHA
        assert report["status"] == "PASS_SUBMISSION_READY_FOR_KAGGLE"
        assert report["run_id"] == RUN_ID
        assert report["contract_sha256"] == CONTRACT_SHA
        validate_artifacts(root, complete["artifacts"])
        assert sha256_file(submission_path) == report["submission_sha256"]
        assert complete["submission_sha256"] == report["submission_sha256"]
        assert int(report["rows"]) == EXPECTED_ROWS
        assert int(report["shards"]) == EXPECTED_SHARDS
        assert report["language_counts"] == dict(EXPECTED_LANGUAGE_COUNTS)
        assert report["qa"]["id_sequence_sha256"] == TEST_ID_SEQUENCE_SHA
        assert report["active_runtime_modified"] is False
        assert report["kaggle_submission_performed"] is False
        return report
    except Exception:
        return None


existing = validate_completed(REPORT_FINAL) if REPORT_FINAL.is_dir() else None
if existing is not None:
    print("\n[reprise] 20.K6 deja termine et verifie :", existing["status"])
else:
    assert not REPORT_FINAL.exists(), (
        "Un dossier final 20.K6 incomplet existe ; le renommer avant de relancer.",
        REPORT_FINAL,
    )
    REPORT_STAGING.mkdir(parents=True, exist_ok=True)
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_json(
        {"contract": CONTRACT, "contract_sha256": CONTRACT_SHA, "run_id": RUN_ID},
        REPORT_STAGING / "contract.json",
    )
    atomic_json(SOFTWARE, REPORT_STAGING / "environment.json")
    atomic_json(portable_inventory, REPORT_STAGING / "test_inventory.json")
    pd.DataFrame(portable_inventory).to_csv(
        REPORT_STAGING / "test_inventory.csv", index=False
    )
    atomic_json(RUNTIME_CONFIG, REPORT_STAGING / "approved_runtime_config_k5.json")

    # Les poids et les LM sont copies une fois vers le SSD local, puis SHA-verifies.
    local_candidates = [Path("/mnt/disks/local-scratch"), Path("/content")]
    required_free = 6 * 2**30 + max(record["bytes"] for record in test_records)
    local_base_root = next(
        (
            candidate
            for candidate in local_candidates
            if candidate.is_dir()
            and os.access(candidate, os.W_OK)
            and shutil.disk_usage(candidate).free >= required_free
        ),
        None,
    )
    assert local_base_root is not None, (
        f"Il faut au moins {required_free / 2**30:.1f} Gio libres sur le SSD local."
    )
    local_asset_root = local_base_root / "kenlm_k6_assets" / RUN_ID
    local_work_root = Path("/content/kenlm_k6_worker") / RUN_ID
    local_stage_root = local_base_root / "kenlm_k6_test_stage" / RUN_ID
    local_asset_root.mkdir(parents=True, exist_ok=True)
    local_work_root.mkdir(parents=True, exist_ok=True)
    local_stage_root.mkdir(parents=True, exist_ok=True)

    local_base = local_asset_root / "base_step1250_bf16.pt"
    local_adapter = local_asset_root / "adapter_mas_step1250.pt"
    local_plan = local_asset_root / "lora_plan.parquet"
    print("\nCopie/verifications des actifs K5 vers le SSD local...")
    copy_atomic_verified(BASE_WEIGHT, local_base, BASE_SHA)
    copy_atomic_verified(ADAPTER_WEIGHT, local_adapter, ADAPTER_SHA)
    copy_atomic_verified(LORA_PLAN, local_plan, PLAN_SHA)

    localized_config = json.loads(json.dumps(RUNTIME_CONFIG))
    localized_assets: dict[tuple[str, str], tuple[str, str]] = {}
    for group, config in localized_config.items():
        key = (config["binary_sha256"], config["unigrams_sha256"])
        if key not in localized_assets:
            asset_root = local_asset_root / config["asset_id"]
            local_binary = asset_root / "model.binary"
            local_unigrams = asset_root / "unigrams.txt"
            copy_atomic_verified(
                PACKAGE_ROOT / config["lm_bin"], local_binary, config["binary_sha256"]
            )
            copy_atomic_verified(
                PACKAGE_ROOT / config["uni_file"],
                local_unigrams,
                config["unigrams_sha256"],
            )
            localized_assets[key] = (str(local_binary), str(local_unigrams))
        config["lm_bin"], config["uni_file"] = localized_assets[key]
    local_runtime_config = local_work_root / "runtime_config.json"
    atomic_json(localized_config, local_runtime_config)

    worker_manifest = [dict(record) for record in test_records]
    worker_manifest_path = local_work_root / "test_manifest.json"
    with open(worker_manifest_path, "w", encoding="utf-8") as handle:
        json.dump(worker_manifest, handle, ensure_ascii=False, indent=2)

    worker_path = REPORT_STAGING / "inference_worker.py"
    worker_path.write_text(K6_WORKER_SOURCE, encoding="utf-8")
    assert sha256_file(worker_path) == CONTRACT["worker_sha256"]
    worker_result_path = local_work_root / "worker_result.json"
    worker_input_path = local_work_root / "worker_input.json"
    worker_log_path = REPORT_STAGING / "worker.log"
    if worker_result_path.exists():
        worker_result_path.unlink()
    worker_spec = {
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "base_parameters": BASE_PARAMETERS,
        "adapter_parameters": ADAPTER_PARAMETERS,
        "total_parameters": TOTAL_PARAMETERS,
        "base_weight": str(local_base),
        "base_weight_sha256": BASE_SHA,
        "adapter_weight": str(local_adapter),
        "adapter_weight_sha256": ADAPTER_SHA,
        "lora_plan": str(local_plan),
        "lora_plan_sha256": PLAN_SHA,
        "lora_plan_rows": LORA_PLAN_ROWS,
        "runtime_config": str(local_runtime_config),
        "runtime_config_sha256": sha256_file(local_runtime_config),
        "runtime_groups": list(RUNTIME_GROUPS),
        "active_test_groups": list(ACTIVE_TEST_GROUPS),
        "test_manifest": str(worker_manifest_path),
        "test_manifest_sha256": sha256_file(worker_manifest_path),
        "cache_root": str(CACHE_ROOT),
        "local_stage_root": str(local_stage_root),
        "batch_size": BATCH_SIZE,
        "partial_flush_every": PARTIAL_FLUSH_EVERY,
        "chunking": CHUNKING,
        "seed": 20260716,
        "model_card_name": f"afrivoices_k6_{RUN_ID[:12]}",
        "expected_labels_sha256": EXPECTED_LABELS_SHA256,
        "max_decode_errors": MAX_DECODE_ERRORS,
    }
    atomic_json(worker_spec, worker_input_path)

    print("\n▶ INFERENCE TEST 20.K6")
    print("GPU          :", torch.cuda.get_device_name(0), "| BF16 | batch=1")
    print("Clips        :", EXPECTED_ROWS, "| shards :", EXPECTED_SHARDS)
    print("Reprise      : relancer la meme cellule apres une coupure")
    print("Estimation   : A100 ~8-20 h ; L4 ~12-30 h, selon les clips longs/KenLM")
    print("Aucune soumission Kaggle ne sera faite automatiquement.\n")

    started = time.monotonic()
    with open(worker_log_path, "a", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            [sys.executable, str(worker_path), str(worker_input_path), str(worker_result_path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            log_handle.flush()
        returncode = process.wait()

    if returncode != 0 or not worker_result_path.is_file():
        attempt = {
            "schema": 1,
            "cell": "20.K6",
            "status": "INCOMPLETE_RESTARTABLE",
            "run_id": RUN_ID,
            "contract_sha256": CONTRACT_SHA,
            "worker_returncode": returncode,
            "cache_root": str(CACHE_ROOT),
            "report_staging": str(REPORT_STAGING),
            "next_step": "Relancer la meme cellule 20.K6.B ; les caches signes seront repris.",
            "active_runtime_modified": False,
            "kaggle_submission_performed": False,
        }
        atomic_json(attempt, K6_ROOT / "LATEST_ATTEMPT.json")
        raise RuntimeError(attempt["next_step"])

    WORKER_RESULT = read_json(worker_result_path)
    assert WORKER_RESULT["status"] == "PASS_CACHE_COMPLETE", WORKER_RESULT
    assert WORKER_RESULT["run_id"] == RUN_ID
    assert WORKER_RESULT["contract_sha256"] == CONTRACT_SHA
    assert int(WORKER_RESULT["shards"]) == EXPECTED_SHARDS
    assert int(WORKER_RESULT["clips"]) == EXPECTED_ROWS
    assert int(WORKER_RESULT["batch_size"]) == BATCH_SIZE
    assert int(WORKER_RESULT["decode_errors_total"]) <= MAX_DECODE_ERRORS
    assert int(WORKER_RESULT["empty_predictions_total"]) <= MAX_EMPTY_PREDICTIONS
    if WORKER_RESULT["model_loaded"]:
        assert int(WORKER_RESULT["max_resident_decoders"]) == 1
        assert WORKER_RESULT["labels_sha256"] == EXPECTED_LABELS_SHA256
    else:
        assert int(WORKER_RESULT["max_resident_decoders"]) == 0
        assert int(WORKER_RESULT["shards_reused"]) == EXPECTED_SHARDS
        assert int(WORKER_RESULT["shards_recomputed"]) == 0
        assert WORKER_RESULT["labels_sha256"] is None
    atomic_json(WORKER_RESULT, REPORT_STAGING / "worker_result.json")

    # Assemblage dans l'ordre exact de l'inventaire test.
    cache_by_relative = {
        record["relative_path"]: record for record in WORKER_RESULT["cache_records"]
    }
    assert set(cache_by_relative) == {record["relative_path"] for record in test_records}
    parts = []
    cache_manifest = []
    expected_ids_cursor = 0
    for record in test_records:
        cache = cache_by_relative[record["relative_path"]]
        cache_path = require_child(cache["output"], CACHE_ROOT)
        meta_path = require_child(cache["meta"], CACHE_ROOT)
        assert sha256_file(cache_path) == cache["cache_sha256"]
        meta = read_json(meta_path)
        assert meta["source_sha256"] == record["source_sha256"]
        assert meta["cache_sha256"] == cache["cache_sha256"]
        frame = pd.read_parquet(cache_path)
        assert list(frame.columns) == ["id", "language", "transcription"]
        ids = frame["id"].astype(str).tolist()
        assert sequence_digest(ids) == record["ids_sha256"]
        assert ids == all_ids[expected_ids_cursor : expected_ids_cursor + len(ids)]
        expected_ids_cursor += len(ids)
        parts.append(frame)
        cache_manifest.append(
            {
                "relative_path": record["relative_path"],
                "source_sha256": record["source_sha256"],
                "cache_sha256": cache["cache_sha256"],
                "cache_meta_sha256": sha256_file(meta_path),
                "rows": len(frame),
                "decode_errors": int(meta.get("decode_errors", 0)),
                "empty_predictions": int(meta.get("empty_predictions", 0)),
                "segmented_clips": int(meta.get("segmented_clips", 0)),
            }
        )
    assert expected_ids_cursor == EXPECTED_ROWS
    submission = pd.concat(parts, ignore_index=True)
    submission["id"] = submission["id"].astype(str)
    submission["language"] = submission["language"].astype(str)
    raw_predictions = submission["transcription"].fillna("").astype(str)
    normalized = raw_predictions.map(normalize_wer_text)
    empty_before = normalized.str.strip().eq("")
    normalized.loc[empty_before] = "a"
    submission["transcription"] = normalized
    submission = submission[["id", "language", "transcription"]]

    # QA bloquante, sans correction humaine.
    assert len(submission) == EXPECTED_ROWS
    assert list(submission.columns) == ["id", "language", "transcription"]
    assert submission["id"].notna().all() and submission["id"].is_unique
    assert submission["id"].tolist() == all_ids
    assert submission["language"].value_counts().to_dict() == dict(EXPECTED_LANGUAGE_COUNTS)
    assert not submission["transcription"].isna().any()
    assert not submission["transcription"].str.strip().eq("").any()
    assert int(empty_before.sum()) <= MAX_EMPTY_PREDICTIONS
    assert submission["transcription"].map(normalize_wer_text).equals(
        submission["transcription"]
    )
    assert submission["transcription"].map(
        lambda value: unicodedata.normalize("NFC", value) == value
    ).all()
    assert not submission["transcription"].str.contains("�", regex=False).any()
    assert not submission["transcription"].map(
        lambda value: any(
            unicodedata.category(character) == "Cc" for character in value
        )
    ).any()
    word_counts = submission["transcription"].str.split().str.len()
    assert int(word_counts.max()) < 500

    word_stats = (
        pd.DataFrame({"language": submission["language"], "words": word_counts})
        .groupby("language")["words"]
        .agg(
            rows="count",
            words_median="median",
            words_p95=lambda values: float(values.quantile(0.95)),
            words_max="max",
        )
        .reset_index()
    )
    submission_path = REPORT_STAGING / f"submission_k6_{RUN_ID}.csv"
    atomic_csv(submission, submission_path)
    roundtrip = pd.read_csv(submission_path, dtype=str, keep_default_na=False)
    assert roundtrip.equals(submission.reset_index(drop=True))

    assert sha256_file(ACTIVE_CONFIG) == ACTIVE_CONFIG_SHA_BEFORE
    submission_sha = sha256_file(submission_path)
    atomic_json(cache_manifest, REPORT_STAGING / "cache_manifest.json")
    atomic_csv(word_stats, REPORT_STAGING / "qa_word_stats.csv")
    qa = {
        "status": "PASS",
        "rows": len(submission),
        "columns": list(submission.columns),
        "ids_unique": True,
        "id_sequence_sha256": sequence_digest(submission["id"].tolist()),
        "language_counts": submission["language"].value_counts().to_dict(),
        "empty_predictions_replaced": int(empty_before.sum()),
        "decode_errors": int(WORKER_RESULT["decode_errors_total"]),
        "segmented_clips": int(WORKER_RESULT["segmented_clips_total"]),
        "maximum_words": int(word_counts.max()),
        "normalization_idempotent": True,
        "utf8_nfc": True,
        "control_characters": 0,
        "replacement_characters": 0,
        "roundtrip_csv_identical": True,
        "submission_sha256": submission_sha,
    }
    atomic_json(qa, REPORT_STAGING / "qa_summary.json")

    report = {
        "schema": 1,
        "cell": "20.K6",
        "status": "PASS_SUBMISSION_READY_FOR_KAGGLE",
        "finished_at": now_iso(),
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "k5_run_id": EXPECTED_K5_RUN_ID,
        "package": str(PACKAGE_ROOT),
        "submission": str(REPORT_FINAL / submission_path.name),
        "submission_sha256": submission_sha,
        "rows": EXPECTED_ROWS,
        "shards": EXPECTED_SHARDS,
        "language_counts": dict(EXPECTED_LANGUAGE_COUNTS),
        "worker": {key: value for key, value in WORKER_RESULT.items() if key != "cache_records"},
        "qa": qa,
        "active_config_sha256_before": ACTIVE_CONFIG_SHA_BEFORE,
        "active_config_sha256_after": sha256_file(ACTIVE_CONFIG),
        "active_runtime_modified": False,
        "kaggle_submission_performed": False,
        "manual_test_transcription_or_correction": False,
        "raw_audio_or_logits_written_to_drive": False,
        "next_step": "Verifier 20.K6.C puis envoyer manuellement ce CSV sur Kaggle.",
    }
    atomic_json(report, REPORT_STAGING / "inference_report_20_K6.json")

    artifact_names = [
        "contract.json", "environment.json", "test_inventory.json",
        "test_inventory.csv", "approved_runtime_config_k5.json",
        "inference_worker.py", "worker.log", "worker_result.json",
        "cache_manifest.json", "qa_word_stats.csv", "qa_summary.json",
        "inference_report_20_K6.json", submission_path.name,
    ]
    artifacts = {
        name: artifact_metadata(REPORT_STAGING / name) for name in artifact_names
    }
    complete = {
        "schema": 1,
        "cell": "20.K6",
        "status": "PASS_SUBMISSION_READY_FOR_KAGGLE",
        "finished_at": now_iso(),
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "artifacts": artifacts,
        "cache_manifest_sha256": sha256_file(REPORT_STAGING / "cache_manifest.json"),
        "submission_sha256": submission_sha,
        "active_runtime_modified": False,
        "kaggle_submission_performed": False,
    }
    atomic_json(complete, REPORT_STAGING / "_COMPLETE.json")
    os.replace(REPORT_STAGING, REPORT_FINAL)
    existing = validate_completed(REPORT_FINAL)
    assert existing is not None

    convenience = REPORT_ROOT / f"submission_k6_{RUN_ID}.csv"
    copy_atomic_verified(REPORT_FINAL / submission_path.name, convenience, submission_sha)

LATEST = {
    "schema": 1,
    "cell": "20.K6",
    "status": "PASS_SUBMISSION_READY_FOR_KAGGLE",
    "run_id": RUN_ID,
    "contract_sha256": CONTRACT_SHA,
    "report": str(REPORT_FINAL / "inference_report_20_K6.json"),
    "report_sha256": sha256_file(REPORT_FINAL / "inference_report_20_K6.json"),
    "complete": str(REPORT_FINAL / "_COMPLETE.json"),
    "complete_sha256": sha256_file(REPORT_FINAL / "_COMPLETE.json"),
    "submission": str(REPORT_FINAL / f"submission_k6_{RUN_ID}.csv"),
    "submission_sha256": sha256_file(REPORT_FINAL / f"submission_k6_{RUN_ID}.csv"),
    "active_runtime_modified": False,
    "kaggle_submission_performed": False,
}
atomic_json(LATEST, K6_ROOT / "LATEST_ATTEMPT.json")
atomic_json(LATEST, K6_ROOT / "LATEST.json")
atomic_json(LATEST, K6_ROOT / "LATEST_PASS.json")

print("\n=== RESULTAT 20.K6 ===")
print("STATUT     :", LATEST["status"])
print("Soumission :", LATEST["submission"])
print("SHA256     :", LATEST["submission_sha256"])
print("Lignes     :", EXPECTED_ROWS, "| IDs uniques : oui")
print("Kaggle     : aucune soumission automatique ; envoi manuel apres 20.K6.C")


## 20.K6.C — Résultat publié (lecture seule)

In [ ]:
# 20.K6.C — Lecture seule du résultat et chemin du CSV
import json
from pathlib import Path
import pandas as pd

root = Path("/content/afrivoices_project/finetune_runs/experiment_run/reports/kenlm_v6_20_K6")
latest_path = root / "LATEST.json"
assert latest_path.is_file(), "20.K6.B n'est pas encore terminé."
latest = json.load(open(latest_path, encoding="utf-8"))
print(json.dumps(latest, indent=2, ensure_ascii=False))

if latest["status"] == "PASS_SUBMISSION_READY_FOR_KAGGLE":
    report_root = Path(latest["report"]).parent
    qa = json.load(open(report_root / "qa_summary.json", encoding="utf-8"))
    display(pd.read_csv(report_root / "qa_word_stats.csv"))
    print("\nQA :", json.dumps(qa, indent=2, ensure_ascii=False))
    print("\n✅ CSV prêt pour un envoi manuel sur Kaggle :")
    print(latest["submission"])
    print("SHA256 :", latest["submission_sha256"])
else:
    print("\n⛔ Le résultat n'est pas PASS. Relancer 20.K6.B ; ne rien soumettre.")
